In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
from os.path import dirname, pardir
from queue import Queue
from pathlib import Path
import enum
from enum import Enum
from typing import Any, Callable, Optional, Generic, TypeVar, Union, cast
__cwd = str(Path().resolve())
__par = __cwd + os.sep + pardir
if __par not in sys.path:
  sys.path.append(__par)
for p in sys.path:
    print(p)
# Import
import importlib
import types
def walk_reload(module: types.ModuleType) -> None:
    if hasattr(module, "__all__"):
        for submodule_name in module.__all__:
            walk_reload(getattr(module, submodule_name))
    importlib.reload(module)
import datetime
import subprocess
from py_modules.logging_lib import setup_logging
from py_modules.lib_aosp_base import *
from py_modules.lib_aosp_testing import *
from py_modules.lib_sh import *
from ci_config import *

logger = setup_logging()
# logger.debug(current_config)
# shell_run3('pwsh -C "while(1) { Write-Host \'Hello World\'; }"')
output_dir = os.path.join(aosp_host_working_dir, f"comp_stress_{datetime.now().strftime('%Y%m%d%H%M%S')}")
print(output_dir)

In [ ]:
# shell_run2('ls')
# shell_run2(f'hello')

In [ ]:
import threading
from typing import Optional, Tuple
import subprocess
from datetime import datetime
import os
import time

def print_output(line):
    print(line)

def check_end_condition(line):
    return "visit_round: 100 end" in line

# Define the configurations for the experiment
configurations = [
    # {"name": "BaseLine1", "stress": "No", "remap": "no", "decomp": "yes(kernel)", "user_comp": "no", "read": 1, "perftime": 20},
    # {"name": "BaseLine4096", "stress": "No", "remap": "no", "decomp": "yes(kernel)", "user_comp": "no", "read": 4096, "perftime": 20},
    # {"name": "BaseLineStress1", "stress": "10MB", "remap": "no", "decomp": "yes(kernel)", "user_comp": "no", "read": 1, "perftime": 200},
    # {"name": "BaseLineStress1Alg", "stress": "No", "remap": "no", "decomp": "yes(kernel)", "user_comp": "no", "read": 1, "perftime": 20, "alg": "yes"},
    # {"name": "BaseLineStress4096", "stress": "10MB", "remap": "no", "decomp": "yes(kernel)", "user_comp": "no", "read": 4096, "perftime": 20},
    # {"name": "InstrOne_1", "stress": "10MB", "remap": "yes", "decomp": "yes", "user_comp": "no", "read": 1, "perftime": 20},
    # {"name": "InstrOne_4096", "stress": "10MB", "remap": "yes", "decomp": "yes", "user_comp": "no", "read": 4096, "perftime": 20},
    {"name": "InstrTwo_1", "stress": "10MB", "remap": "yes", "decomp": "no", "user_comp": "no", "read": 1, "perftime": 20},
    # {"name": "InstrTwo_4096", "stress": "10MB", "remap": "yes", "decomp": "no", "user_comp": "no", "read": 4096, "perftime": 20},
    # {"name": "InstrThree_1_perf15", "stress": "10MB", "remap": "yes", "decomp": "no", "user_comp": "yes", "read": 1, "perftime": 20},
    # {"name": "InstrThree_1_perf20", "stress": "10MB", "remap": "yes", "decomp": "no", "user_comp": "yes", "read": 1, "perftime": 20},
    # {"name": "InstrThree_1_perf25", "stress": "10MB", "remap": "yes", "decomp": "no", "user_comp": "yes", "read": 1, "perftime": 20},
    # {"name": "InstrThree_1_perf60", "stress": "10MB", "remap": "yes", "decomp": "no", "user_comp": "yes", "read": 1, "perftime": 60},
    # {"name": "InstrThree_4096", "stress": "10MB", "remap": "yes", "decomp": "no", "user_comp": "yes", "read": 4096, "perftime": 60}
]

use_simple_perf = False

def get_zram_stat():
    # Fetch zram stats
    cmd = "adb -s {} shell 'cat /sys/block/zram0/stat'".format(serial)
    result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return result.stdout.decode('utf-8')

def get_comp_stress_info():
    # Fetch comp_stress info
    cmd = "adb -s {} shell 'ps -A -o pid,minfl,majfl,rss,swap,name | grep comp_stress'".format(serial)
    result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return result.stdout.decode('utf-8')

def run_experiment(config, output_dir):
    # shell_run(f"adb -s {serial} shell killall comp_stress_test", None, False, False, print_output)
    cmd = f"/system/bin/comp_stress_test -tf 2 -numt 0 -nd"
    if config["remap"] == "no":
        cmd += " -nr"
    if config["decomp"] == "no":
        cmd += " -ndp"
    if config["user_comp"] == "no":
        cmd += " -nc"
        cmd += " -nd"
    if config["stress"] != "No":
        cmd += " -mcg"
    if "alg" in config and config["alg"] == "yes":
        cmd += " -alg"
    cmd += f" -bs {config['read']}"
    if use_simple_perf:
        perf_time = config["perftime"]
        sched_events = "sched:sched_process_wait,sched:sched_stat_sleep,sched:sched_stat_iowait,sched:sched_stat_blocked,sched:sched_switch"
        events = f"cpu-cycles,task-clock,context-switches,instructions,page-faults,major-faults,minor-faults,{sched_events}"
        device_cmd = f"cd /data/local/tmp/ && simpleperf record --call-graph fp -e {events} --duration {perf_time} -- {cmd}"
    else:
        device_cmd = cmd
    full_cmd = f"adb -s {serial} shell \"sh -c \'{device_cmd}\'\""
    if True:

        if True and config["stress"] != "No":
            stress_cmd = r"setup_zram.py; zram_insmod.py; test_compressfd_stress.ps1"
            # Execute the stress_cmd
            subprocess.run(stress_cmd, shell=True)
        else:
            stress_cmd = r"setup_zram.py;"
            # Execute the stress_cmd
            subprocess.run(stress_cmd, shell=True)
            
        
        if "BaseLine" in config['name']:
            remap_cmd = f"adb -s {serial} shell 'echo 0 > /proc/lxr_force_use_remap'"
            subprocess.run(remap_cmd, shell=True, check=False)
        else:
            remap_cmd = f"adb -s {serial} shell 'echo 1 > /proc/lxr_force_use_remap'"
            subprocess.run(remap_cmd, shell=True, check=True)
        


    with open(os.path.join(output_dir, f"{config['name']}_stdout.txt"), 'w') as f:
        def callback(line):
            f.write(line + "\n")
            if check_end_condition(line) == True:
                # Write zram stat and comp_stress info to the file
                f.write("100 run reached\n")
                f.write("Zram stat:\n")
                f.write("readIO\treadMerge\treadSectors\treadTicks\twriteIO\twriteMerge\twriteSectors\twriteTicks\tinFlight\tioTicks\ttime_in_queue\n")
                f.write(get_zram_stat() + "\n")
                f.write("Comp_stress info:\n")
                f.write("  pid,  minfl, majfl,  rss,  swap, comm\n")
                f.write(get_comp_stress_info() + "\n")
                f.write("End of comp_stress info\n")
            print(line)
            return check_end_condition(line) == False

        print(config['name'], full_cmd)
        shell_run(full_cmd, None, False, True, callback)
        if use_simple_perf:
            pf_file = os.path.join(output_dir, f"perf_{config['name']}.data")
            pf_ret = os.path.join(output_dir, f"report_{config['name']}.html")
            shell_run(f"adb -s {serial} pull /data/local/tmp/perf.data {pf_file}")
            shell_run(f"python {ASRCDIR}/system/extras/simpleperf/scripts/report_html.py --no_browser -i {pf_file} -o {pf_ret}")


# Create directory for output
os.makedirs(output_dir, exist_ok=True)
for config in configurations:
    run_experiment(config, output_dir)

